In [ ]:
# ============================================
# 1. IMPORTACIONES Y CONFIGURACIÓN DEL SISTEMA
# ============================================
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A  # Librería profesional para augmentations
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm  # Barra de progreso
import torchvision
import segmentation_models_pytorch as smp


# Configuración del dispositivo (Libertad de elección)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Usando dispositivo: {DEVICE}")

# PATHS

In [ ]:
images_path = "png_images/IMAGES"
masks_path = "png_masks/MASKS"
labels_csv_path = "labels.csv"

# CONFIGURACIONES INICIALES

In [ ]:
# Parámetros del modelo
IMG_HEIGHT = 384
IMG_WIDTH = 384
IMG_CHANNELS = 3
BATCH_SIZE = 4  # Reducido por la mayor resolución
EPOCHS = 100
LEARNING_RATE = 1e-4
PATIENCE = 10

In [ ]:
# ============================================
# 2. MAPEO DE ETIQUETAS (59 → 6 clases + ignore)
# ============================================
def create_label_mapping(csv_path):
    """Crea mapeo de las 59 clases originales a 6 clases + ignore"""
    labels_df = pd.read_csv(csv_path)
    
    # Mapeo semántico (texto → clase limpia)
    mapping = {
        # --- TOPWEAR (tops + outerwear juntos) ---
        "blazer": "topwear", "jacket": "topwear", "coat": "topwear",
        "cape": "topwear", "cardigan": "topwear", "suit": "topwear",
        "t-shirt": "topwear", "shirt": "topwear", "top": "topwear",
        "sweater": "topwear", "hoodie": "topwear", "blouse": "topwear",
        "bodysuit": "topwear", "sweatshirt": "topwear", "vest": "topwear", 
        "dress": "topwear",
        
        # --- LOWERWEAR ---
        "pants": "lowerwear", "jeans": "lowerwear", "leggings": "lowerwear",
        "skirt": "lowerwear", "shorts": "lowerwear", "panties": "lowerwear",
        "tights": "lowerwear",
    
        
        # --- FOOTWEAR ---
        "boots": "footwear", "sneakers": "footwear", "flats": "footwear",
        "heels": "footwear", "loafers": "footwear", "sandals": "footwear",
        "wedges": "footwear", "pumps": "footwear", "clogs": "footwear",
        "shoes": "footwear",
        
        # --- BODY ---
        "hair": "body", "skin": "body",
        
        # --- BACKGROUND ---
        "background": "background"
    }
    
    # Aplicar mapeo (todo lo no definido → ignore)
    labels_df["clean_label"] = labels_df["label_list"].map(mapping).fillna("ignore")
    
    # Mapeo numérico final
    numeric_mapping = {
        "background": 0,
        "topwear": 1,
        "lowerwear": 2,
        "footwear": 3,
        "body": 4,
        "ignore": 255  # Para ignorar durante el entrenamiento
    }
    
    labels_df["numeric_label"] = labels_df["clean_label"].map(numeric_mapping)
    
    # Crear diccionario de mapeo original → nuevo
    original_to_new = dict(zip(labels_df.index, labels_df["numeric_label"]))
    
    # Información de clases
    class_names = ["background", "topwear", "lowerwear", "footwear", "body"]
    num_classes = len(class_names)  # 6 clases (excluyendo ignore)
    
    print("Distribución de clases reducidas:")
    print(labels_df["numeric_label"].value_counts().sort_index())
    print(f"\nClases: {class_names}")
    print(f"Número de clases: {num_classes}")
    
    return original_to_new, class_names, num_classes

# Cargar mapeo de etiquetas
label_mapping, CLASS_NAMES, NUM_CLASSES = create_label_mapping(labels_csv_path)

# Colores para visualización
CLASS_COLORS = [
    [0, 0, 0],        # 0 background → negro
    [220, 20, 60],    # 1 topwear → rojo
    [30, 144, 255],   # 2 lowerwear → azul
    [138, 43, 226],   # 4 footwear → violeta
    [46, 139, 87],    # 5 body → verde
]


In [ ]:
# ============================================
# 3. PREPARACIÓN DEL DATAFRAME (SPLIT: 80/10/10)
# ============================================
def create_df(images_dir, masks_dir):
    image_names = sorted(os.listdir(images_dir))
    mask_names = sorted(os.listdir(masks_dir))
    
    image_paths = [os.path.join(images_dir, fname) for fname in image_names]
    mask_paths = [os.path.join(masks_dir, fname) for fname in mask_names]
    
    if len(image_paths) != len(mask_paths):
        print(f"⚠️ ADVERTENCIA: {len(image_paths)} imágenes vs {len(mask_paths)} máscaras.")
    
    return pd.DataFrame({'image_path': image_paths, 'mask_path': mask_paths})

# Generar DataFrames
try:
    full_df = create_df("png_images/IMAGES", "png_masks/MASKS")
    
    # 1. Separar Train (80%) del resto (20%)
    train_df, temp_df = train_test_split(full_df, test_size=0.2, random_state=42)
    
    # 2. Separar el resto en Val (10%) y Test (10%)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    print(f"✅ DataFrames creados exitosamente:")
    print(f"   - Total muestras: {len(full_df)}")
    print(f"   - Train (Entrenamiento): {len(train_df)} imágenes")
    print(f"   - Val   (Ajuste params): {len(val_df)} imágenes")
    print(f"   - Test  (Evaluación final): {len(test_df)} imágenes")

except FileNotFoundError:
    print("❌ ERROR: Rutas no encontradas. Verifica png_images/IMAGES")

In [ ]:
# ============================================
# 4. TRANSFORMACIONES (384x384 HD)
# ============================================
def get_transforms(mode="train"):
    if mode == "train":
        return A.Compose([
            A.Resize(IMG_HEIGHT, IMG_WIDTH),
            A.HorizontalFlip(p=0.5),
            # Aumentación geométrica con zoom más agresivo
            A.ShiftScaleRotate(shift_limit=0.05,scale_limit=0.2,rotate_limit=10,p=0.5),
            A.Perspective(scale=(0.03, 0.06), p=0.2),  
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
            A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),  # Ruido leve
            A.Blur(blur_limit=3, p=0.4),  # Desenfoque ocasional
            A.CoarseDropout(max_holes=6, max_height=30, max_width=30, 
                           min_holes=2, min_height=10, min_width=10, 
                           fill_value=128,  mask_fill_value=0, p=0.5),  # Tapar zonas pequeñas (oclusión)    
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(IMG_HEIGHT, IMG_WIDTH),    
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


In [ ]:
# ============================================
# 5. DATASET PERSONALIZADO (CORREGIDO)
# ============================================
class ClothingDataset(Dataset):
    def __init__(self, df, transform=None, label_mapping=None):
        self.df = df
        self.transform = transform
        
        if label_mapping:
            max_id = max(label_mapping.keys())
            # Inicializamos con 255 (Ignore) por defecto para seguridad
            self.mapping_array = np.full(max_id + 1, 255, dtype=np.uint8)
            for k, v in label_mapping.items():
                self.mapping_array[k] = v
        else:
            self.mapping_array = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Cargar máscara (asegurar que sea un solo canal)
        mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
        
        # 1. Manejo seguro de valores fuera de rango (Evitar el Bug "Wedges")
        if self.mapping_array is not None:
            # Primero: Todo lo que esté fuera del rango de tus etiquetas (0-58)
            # lo marcamos como una etiqueta segura temporal (ej. 0 background)
            # O simplemente aplicamos el mapeo solo a los válidos.
            
            # Opción robusta: Usar un array de mapeo extendido que maneje 255
            # Pero para seguir tu lógica simple:
            h, w = mask.shape
            mask_mapped = np.zeros((h, w), dtype=np.uint8) # Por defecto fondo
            
            # Solo mapeamos los píxeles que están dentro del rango conocido (0-58)
            valid_pixels = mask < len(self.mapping_array)
            mask_mapped[valid_pixels] = self.mapping_array[mask[valid_pixels]]
            
            # Los píxeles >= 59 (como el 255) se quedan en 0 (Background) 
            # o podemos ponerles 255 explícitamente si preferimos ignorar.
            # Siguiendo tu lógica de estabilidad:
            mask = mask_mapped

        # 2. TRUCO DE ESTABILIDAD: Convertir Ignore (255) a Background (0)
        # Ahora es redundante si inicializamos mask_mapped en 0, pero lo dejamos por seguridad.
        mask[mask == 255] = 0
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        return image, mask.long()

# VISUALIZACIONES

In [ ]:
# ============================================
# 6. VISUALIZACIÓN DE MÁSCARAS (LEYENDA POR FILA)
# ============================================
import matplotlib.patches as mpatches

def visualize_augmentations(dataset, class_names, class_colors, num_images=4):
    # Dataloader temporal
    loader = DataLoader(dataset, batch_size=num_images, shuffle=True)
    images, masks = next(iter(loader))
    
    # Configuración de des-normalización
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    # Crear figura: Altura dinámica según número de imágenes
    fig, axes = plt.subplots(num_images, 3, figsize=(18, 5 * num_images))
    
    # Si solo pides 1 imagen, axes no es una matriz, así que lo forzamos a serlo
    if num_images == 1:
        axes = np.expand_dims(axes, axis=0)
    
    # Preparamos los "Patches" de la leyenda (se reutilizan en cada fila)
    legend_patches = []
    for name, color in zip(class_names, class_colors):
        c_norm = [c/255.0 for c in color]
        legend_patches.append(mpatches.Patch(color=c_norm, label=name))
    
    for i in range(num_images):
        # --- 1. Imagen Original ---
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        # --- 2. Máscara Coloreada ---
        msk = masks[i].cpu().numpy()
        msk_rgb = np.zeros((msk.shape[0], msk.shape[1], 3))
        
        # Pintar solo las clases presentes en esta máscara específica
        # (Opcional: puedes pintar todas, pero así es más eficiente)
        unique_classes = np.unique(msk)
        for class_idx in unique_classes:
             if class_idx < len(class_colors): # Protección por seguridad
                color = class_colors[class_idx]
                msk_rgb[msk == class_idx] = [c/255.0 for c in color]
        
        # --- PLOTEAR ---
        # Columna 1: Original
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original {i+1}")
        axes[i, 0].axis("off")
        
        # Columna 2: Ground Truth
        axes[i, 1].imshow(msk_rgb)
        axes[i, 1].set_title("Máscara Real")
        axes[i, 1].axis("off")
        
        # Columna 3: Overlay + LEYENDA
        axes[i, 2].imshow(img)
        axes[i, 2].imshow(msk_rgb, alpha=0.5)
        axes[i, 2].set_title("Superposición")
        axes[i, 2].axis("off")
        
        # AQUÍ ESTÁ EL CAMBIO: Leyenda en cada fila
        axes[i, 2].legend(
            handles=legend_patches, 
            bbox_to_anchor=(1.05, 1), 
            loc='upper left', 
            borderaxespad=0.,
            title="Clases"
        )

    plt.tight_layout()
    plt.show()

# --- EJECUCIÓN ---
if 'train_df' in locals():
    # Creamos dataset visualización
    viz_ds = ClothingDataset(
        train_df, 
        transform=get_transforms("train"), 
        label_mapping=label_mapping
    )
    
    print("🎨 Generando visualización fila por fila...")
    # Puedes cambiar num_images a 8 o 10 y verás la leyenda en cada uno
    visualize_augmentations(viz_ds, CLASS_NAMES, CLASS_COLORS, num_images=4)
else:
    print("⚠️ Falta cargar el DataFrame.")

# ARQUITECTURA

In [ ]:
# ============================================
# 7. CONFIGURACIÓN DEL MODELO: U-Net++ HD
# ============================================
import segmentation_models_pytorch as smp
import torch.nn as nn

# 1. MODELO
# Usamos U-Net++ (PlusPlus) que tiene "puentes" densos. 
# Esto es CRÍTICO para que el modelo conecte la parte de arriba del vestido con la de abajo.
model = smp.UnetPlusPlus(
    encoder_name="resnet34",        # Backbone potente y equilibrado
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=NUM_CLASSES,
    decoder_attention_type="scse" 
)
model.to(DEVICE)

# 2. PESOS (Estrategia Anti-Confusión)
# Mantenemos la presión sobre los vestidos para que no los confunda.
# [Background, Topwear, Lowerwear, Footwear, Body]
class_weights = torch.tensor([0.5, 2.5, 3.0, 2.5, 3.0]).to(DEVICE)
#class_weights = torch.tensor([1.0, 1.5, 1.0, 2.0, 3.0]).to(DEVICE) mejor funcionamiento

#class_weights = torch.tensor([1.5, 2.0, 1.0, 2.0, 2.5]).to(DEVICE)
#class_weights = torch.tensor([1.5, 1.5, 1.0, 2.5, 2.0]).to(DEVICE)

# 3. LOSS (Weighted CE + Dice)
# 1. Focal Loss ayuda con ejemplos difíciles (como vestidos largos)
focal_loss = smp.losses.FocalLoss(mode='multiclass')
# 2. Dice Loss es vital para el solapamiento
dice_loss = smp.losses.DiceLoss(mode='multiclass', from_logits=True, ignore_index=255)
# 3. CE con pesos
ce_loss = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)

def criterion(pred, target):
    # Damos un peso fuerte a la combinación
    return (0.4 * ce_loss(pred, target)) + (0.3 * dice_loss(pred, target)) + (0.3 * focal_loss(pred, target))

loss_fn = criterion 

# 4. Optimizador
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# 5. Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

# 6. Scaler
scaler = torch.cuda.amp.GradScaler()

print(f"✅ Modelo RTX Configurado: U-Net++ (ResNet34) @ 384x384.")

In [ ]:
# ============================================
# 8. FUNCIONES DE ENTRENAMIENTO MEJORADAS (CON IoU)
# ============================================

def get_metrics_batch(pred_logits, targets, num_classes=5):
    """Calcula IoU Score promedio del batch (La métrica de la verdad)"""
    # 1. Convertir Logits a Probabilidades (Softmax)
    probs = torch.softmax(pred_logits, dim=1)
    
    # 2. Convertir a Predicciones One-Hot
    preds = torch.argmax(probs, dim=1)
    
    # Usamos la función optimizada de la librería smp
    # tp: true positives, fp: false positives, fn: false negatives, tn: true negatives
    tp, fp, fn, tn = smp.metrics.get_stats(
        preds, 
        targets, 
        mode='multiclass', 
        num_classes=num_classes,
        ignore_index=255
    )
    
    # IoU = TP / (TP + FP + FN)
    iou_score = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
    accuracy = smp.metrics.accuracy(tp, fp, fn, tn, reduction="micro")
    
    return iou_score, accuracy

def train_one_epoch(loader, model, optimizer, loss_fn, scaler):
    model.train()
    running_loss = 0.0
    running_iou = 0.0  # <--- Métrica nueva
    
    loop = tqdm(loader, desc="Entrenando", leave=True)
    
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.to(device=DEVICE)

        # Forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # Backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Calcular Métricas (Ahora medimos IoU)
        iou, acc = get_metrics_batch(predictions, targets, num_classes=NUM_CLASSES)
        
        running_loss += loss.item()
        running_iou += iou
        
        # Mostramos IoU en la barra (esto es lo que debes vigilar)
        loop.set_postfix(loss=f"{loss.item():.4f}", iou=f"{iou:.4f}")

    return running_loss / len(loader), running_iou / len(loader)

def validate_one_epoch(loader, model, loss_fn):
    model.eval()
    running_loss = 0.0
    running_iou = 0.0
    
    loop = tqdm(loader, desc="Validando", leave=True)
    
    with torch.no_grad():
        for batch_idx, (data, targets) in enumerate(loop):
            data = data.to(device=DEVICE)
            targets = targets.to(device=DEVICE)

            predictions = model(data)
            loss = loss_fn(predictions, targets)
            
            iou, acc = get_metrics_batch(predictions, targets, num_classes=NUM_CLASSES)

            running_loss += loss.item()
            running_iou += iou
            loop.set_postfix(loss=f"{loss.item():.4f}", iou=f"{iou:.4f}")
            
    return running_loss / len(loader), running_iou / len(loader)

In [ ]:
# ============================================
# 9. EJECUCIÓN MAESTRA (CON IoU REAL)
# ============================================

# --- HIPERPARÁMETROS (Por si no estaban definidos) ---
BATCH_SIZE = 4      # Ajusta según tu VRAM (4 para 8GB, 8 para 16GB)
EPOCHS = 30         # Suficiente gracias al Early Stopping
PATIENCE = 6        # Esperamos 6 épocas antes de rendirnos

# 1. Dataloaders (Optimizados)
# num_workers=2 suele ser más rápido que 0 si tienes buena CPU, si falla ponlo en 0
train_ds = ClothingDataset(train_df, transform=get_transforms("train"), label_mapping=label_mapping)
val_ds = ClothingDataset(val_df, transform=get_transforms("val"), label_mapping=label_mapping)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# 2. Historial (Ahora guardamos IoU)
history = {'train_loss': [], 'train_iou': [], 'val_loss': [], 'val_iou': []}

# Variables de control
epochs_no_improve = 0
best_val_loss = float('inf')

print(f"🚀 Iniciando entrenamiento U-Net++ @ 384x384")
print(f"   - Objetivo: Maximizar IoU (Intersección sobre Unión)")
print(f"   - Estrategia: Body Weight aumentado para recuperar piel")

for epoch in range(EPOCHS):
    print(f"\n📢 Época {epoch + 1}/{EPOCHS}")
    
    # --- Entrenar y Validar (Recibimos IoU) ---
    t_loss, t_iou = train_one_epoch(train_loader, model, optimizer, loss_fn, scaler)
    v_loss, v_iou = validate_one_epoch(val_loader, model, loss_fn)
    
    # --- Guardar historial ---
    history['train_loss'].append(t_loss)
    history['train_iou'].append(t_iou)
    history['val_loss'].append(v_loss)
    history['val_iou'].append(v_iou)
    
    # --- Reporte Honesto ---
    # IoU > 50% es decente, > 70% es excelente, > 80% es estado del arte.
    print(f"   📉 Train Loss: {t_loss:.4f} | 🏆 Train IoU: {t_iou*100:.2f}%")
    print(f"   📉 Val Loss:   {v_loss:.4f} | 🏆 Val IoU:   {v_iou*100:.2f}%")
    
    # --- Scheduler ---
    scheduler.step(v_loss)
    
    # --- EARLY STOPPING ---
    if v_loss < best_val_loss:
        diff = best_val_loss - v_loss
        best_val_loss = v_loss
        epochs_no_improve = 0 
        # Guardamos el modelo
        torch.save(model.state_dict(), "mejor_modelo_384_fixed_body1.pth")
        print(f"   💾 ¡MEJORA DETECTADA! Modelo guardado (Loss bajó -{diff:.4f})")
    else:
        epochs_no_improve += 1
        print(f"   ⏳ Sin mejora en {epochs_no_improve}/{PATIENCE} épocas.")
        
        if epochs_no_improve >= PATIENCE:
            print(f"\n🛑 PARADA TEMPRANA: El modelo se estancó. Entrenamiento terminado.")
            break

print("\n🏁 Entrenamiento finalizado. ¡Hora de probar!")

# GRÁFICAS

In [ ]:
# ============================================
# 10. VALIDACIÓN VISUAL (LA PRUEBA DE FUEGO)
# ============================================
def visualize_predictions(model, loader, num_images=4):
    model.eval()
    
    # 1. Obtener un batch de validación
    images, masks = next(iter(loader))
    images = images.to(DEVICE)
    
    # 2. Predecir
    with torch.no_grad():
        logits = model(images)
        # Convertimos logits a etiquetas (0, 1, 2, 3, 4)
        preds = torch.argmax(logits, dim=1)
        
    # 3. Preparar para visualización (Mover a CPU)
    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()
    
    # Des-normalización para ver la foto original bien
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    # 4. Plot
    fig, axes = plt.subplots(num_images, 3, figsize=(15, 5 * num_images))
    if num_images == 1: axes = np.expand_dims(axes, axis=0)

    for i in range(num_images):
        # -- Imagen Original --
        img = images[i].permute(1, 2, 0).numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        # -- Ground Truth (Real) --
        # Coloreamos usando tu paleta CLASS_COLORS
        gt_mask = masks[i].numpy()
        gt_color = np.zeros((gt_mask.shape[0], gt_mask.shape[1], 3))
        for cls_idx in range(len(CLASS_COLORS)):
            gt_color[gt_mask == cls_idx] = [c/255.0 for c in CLASS_COLORS[cls_idx]]
            
        # -- Predicción (Modelo) --
        pr_mask = preds[i].numpy()
        pr_color = np.zeros((pr_mask.shape[0], pr_mask.shape[1], 3))
        for cls_idx in range(len(CLASS_COLORS)):
            pr_color[pr_mask == cls_idx] = [c/255.0 for c in CLASS_COLORS[cls_idx]]

        # Mostrar
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Original")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(gt_color)
        axes[i, 1].set_title("Realidad (Etiqueta)")
        axes[i, 1].axis("off")
        
        axes[i, 2].imshow(img)
        axes[i, 2].imshow(pr_color, alpha=0.6) # Superpuesto
        axes[i, 2].set_title("Predicción U-Net++")
        axes[i, 2].axis("off")
        
    plt.tight_layout()
    plt.show()

# --- Ejecutar ---
print("🔍 Inspeccionando resultados en Validación...")
visualize_predictions(model, val_loader, num_images=4)

In [ ]:
# ============================================
# 10. GRÁFICAS DE RENDIMIENTO (CORREGIDO: IoU)
# ============================================
import matplotlib.pyplot as plt

# Verificar qué llaves tenemos realmente (para depuración)
print(f"Llaves disponibles en history: {history.keys()}")

plt.figure(figsize=(15, 6))

# --- GRÁFICA 1: PÉRDIDA (LOSS) ---
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='salmon')
plt.plot(history['val_loss'], label='Val Loss', color='red', linestyle='--', linewidth=2)
plt.title('Curva de Aprendizaje: PÉRDIDA (Debe bajar)')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# --- GRÁFICA 2: IoU (CALIDAD REAL) ---
# Cambiamos 'acc' por 'iou'
plt.subplot(1, 2, 2)
plt.plot(history['train_iou'], label='Train IoU', color='lightgreen')
plt.plot(history['val_iou'], label='Val IoU', color='green', linestyle='--', linewidth=2)
plt.title('Curva de Aprendizaje: IoU (Debe subir)')
plt.xlabel('Épocas')
plt.ylabel('IoU Score (0.0 - 1.0)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 11. INFERENCIA (PRUEBA FINAL CON IMAGEN REAL)
# ============================================
import cv2
import matplotlib.pyplot as plt
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

# 1. Cargar el MEJOR modelo guardado
best_model_path = "mejor_modelo_384_fixed_body1.pth" # El nombre que pusimos en el paso 9
model.load_state_dict(torch.load(best_model_path))
model.eval()
print("✅ Modelo cargado y listo para inferencia.")

# 2. Función de Predicción
def predict_image(image_path, model, device):
    # Leer imagen
    image = cv2.imread(image_path)
    if image is None: return print("❌ Error: No se encuentra la imagen.")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_size = image.shape[:2] # Guardamos tamaño original para luego
    
    # Transformación (Solo Resize y Normalización, SIN rotaciones)
    transform = A.Compose([
        A.Resize(384, 384),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    augmented = transform(image=image)["image"]
    input_tensor = augmented.unsqueeze(0).to(device) # Añadir batch dim [1, 3, 384, 384]
    
    # Predecir
    with torch.no_grad():
        logits = model(input_tensor)
        pred_mask = torch.argmax(logits, dim=1).squeeze().cpu().numpy()
        
    # Redimensionar máscara al tamaño original de la foto
    pred_mask_resized = cv2.resize(pred_mask.astype('uint8'), (original_size[1], original_size[0]), interpolation=cv2.INTER_NEAREST)
    
    return image, pred_mask_resized

# 3. Visualizar Resultado
def show_result(image, mask, class_colors):
    plt.figure(figsize=(12, 6))
    
    # Imagen Original
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Imagen Original")
    plt.axis("off")
    
    # Máscara Superpuesta
    # Crear imagen RGB para la máscara
    mask_rgb = np.zeros_like(image)
    for cls_idx, color in enumerate(class_colors):
        mask_rgb[mask == cls_idx] = color
        
    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.imshow(mask_rgb, alpha=0.6) # Transparencia
    plt.title("Segmentación Predicha")
    plt.axis("off")
    
    # Leyenda manual
    patches = [plt.Rectangle((0,0),1,1, color=[c/255 for c in color]) for color in class_colors]
    plt.legend(patches, ["Background", "Topwear", "Lowerwear", "Footwear", "Body"], loc='upper right')
    
    plt.show()

# --- EJECUTAR PRUEBA ---
# Elige una imagen de tu carpeta de validación o una nueva
test_image_path = val_df.iloc[19]['image_path'] # Probamos con la imagen #5 del set de validación

print(f"📸 Probando con: {test_image_path}")
orig_img, pred_mask = predict_image(test_image_path, model, DEVICE)
show_result(orig_img, pred_mask, CLASS_COLORS)

In [ ]:
# ============================================
# 11. PRUEBA DE PREDICCIÓN VISUAL
# ============================================
import matplotlib.patches as mpatches

def predict_and_plot(model, loader, class_names, class_colors, num_samples=3):
    # 1. Cargar el mejor estado del modelo guardado
    # Esto asegura que uses la mejor versión, no la última época (que podría estar overfitteada)
    checkpoint_path = "mejor_modelo_384_fixed_body1.pth"
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path))
        print("✅ Mejor modelo cargado exitosamente.")
    else:
        print("⚠️ No se encontró checkpoint. Usando modelo actual de memoria.")

    model.eval()
    
    # 2. Obtener un batch de datos de validación
    images, masks = next(iter(loader))
    images = images.to(DEVICE)
    
    # 3. Generar predicciones
    with torch.no_grad():
        outputs = model(images)
        # Convertir probabilidades a índice de clase (Argmax)
        preds = torch.argmax(outputs, dim=1)
        
    # Mover a CPU para visualizar
    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()
    
    # Configuración para des-normalizar la imagen (para que se vea natural)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    # 4. Loop de visualización
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5 * num_samples))
    
    # Crear parches para la leyenda
    legend_patches = [mpatches.Patch(color=[c/255.0 for c in color], label=name) 
                      for name, color in zip(class_names, class_colors)]
    
    for i in range(min(num_samples, len(images))):
        # --- A. Imagen Original ---
        img = images[i].permute(1, 2, 0).numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        # --- B. Máscara Real (Ground Truth) ---
        msk_true = masks[i].numpy()
        msk_true_rgb = np.zeros((msk_true.shape[0], msk_true.shape[1], 3))
        for cls_idx in np.unique(msk_true):
            if cls_idx < len(class_colors): # Ignorar valor 255
                msk_true_rgb[msk_true == cls_idx] = [c/255.0 for c in class_colors[cls_idx]]
        
        # --- C. Máscara Predicha ---
        msk_pred = preds[i].numpy()
        msk_pred_rgb = np.zeros((msk_pred.shape[0], msk_pred.shape[1], 3))
        for cls_idx in np.unique(msk_pred):
            msk_pred_rgb[msk_pred == cls_idx] = [c/255.0 for c in class_colors[cls_idx]]
            
        # --- D. Overlay (Predicción sobre Imagen) ---
        
        # --- PLOTEAR ---
        # 1. Original
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original {i+1}")
        axes[i, 0].axis("off")
        
        # 2. Real
        axes[i, 1].imshow(msk_true_rgb)
        axes[i, 1].set_title("Real (Ground Truth)")
        axes[i, 1].axis("off")
        
        # 3. Predicha
        axes[i, 2].imshow(msk_pred_rgb)
        axes[i, 2].set_title("Predicción AI")
        axes[i, 2].axis("off")
        
        # 4. Overlay
        axes[i, 3].imshow(img)
        axes[i, 3].imshow(msk_pred_rgb, alpha=0.5)
        axes[i, 3].set_title("Superposición")
        axes[i, 3].axis("off")
        
        # Leyenda (solo en la última columna de la primera fila para no saturar)
        if i == 0:
            axes[i, 3].legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left', title="Clases")

    plt.tight_layout()
    plt.show()

# --- EJECUTAR ---
# Puedes cambiar num_samples para ver más o menos fotos
predict_and_plot(model, val_loader, CLASS_NAMES, CLASS_COLORS, num_samples=4)

In [ ]:
# ============================================
# 13. MATRIZ DE CONFUSIÓN DETALLADA
# ============================================
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def plot_confusion_matrix(model, loader, class_names):
    model.eval()
    all_preds = []
    all_targets = []
    
    print("🔄 Generando predicciones para matriz de confusión...")
    
    with torch.no_grad():
        for images, masks in tqdm(loader):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)
            
            # Predicción
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            # Aplanar datos (Pixel a Pixel)
            # Solo tomamos píxeles que NO son background (0) para ver mejor la confusión entre ropas
            # O si prefieres ver todo, quitamos el filtro.
            # Aquí analizamos TODO incluyendo fondo:
            preds_flat = preds.cpu().numpy().flatten()
            targets_flat = masks.cpu().numpy().flatten()
            
            # Acumular (Para evitar overflow de memoria, usamos muestreo si es gigante)
            # Para tu MX250 y dataset mediano, podemos intentar acumular todo, 
            # pero si falla, reduciremos la cantidad.
            all_preds.extend(preds_flat)
            all_targets.extend(targets_flat)

    # Convertir a numpy
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    # Generar Matriz
    print("📊 Calculando matriz (esto puede tardar unos segundos)...")
    cm = confusion_matrix(all_targets, all_preds, labels=range(len(class_names)))
    
    # Normalizar por filas (para ver porcentajes de error)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plotear
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicción del Modelo')
    plt.ylabel('Realidad (Ground Truth)')
    plt.title('Matriz de Confusión Normalizada (%)')
    plt.show()
    
    # Reporte numérico
    print("\n📋 Reporte de Clasificación:")
    print(classification_report(all_targets, all_preds, target_names=class_names))

# Ejecutar
plot_confusion_matrix(model, val_loader, CLASS_NAMES)

In [ ]:
# ============================================
# 14. PREDICCIÓN CON TTA (TEST TIME AUGMENTATION)
# ============================================
def predict_with_tta(model, image_tensor):
    """
    Realiza predicción promediando la imagen original y su versión espejada.
    Suaviza errores y mejora bordes.
    """
    model.eval()
    
    # 1. Predicción Normal
    with torch.no_grad():
        logits_normal = model(image_tensor.to(DEVICE))
        
        # 2. Predicción Espejada (Flip Horizontal)
        img_flipped = torch.flip(image_tensor, dims=[3]) # Flip en eje W
        logits_flipped = model(img_flipped.to(DEVICE))
        
        # Volteamos de nuevo la predicción para que coincida con la original
        logits_flipped_back = torch.flip(logits_flipped, dims=[3])
        
        # 3. Promedio
        logits_avg = (logits_normal + logits_flipped_back) / 2.0
        
        # 4. Argmax final
        pred_mask = torch.argmax(logits_avg, dim=1)
        
    return pred_mask.cpu()

# --- VISUALIZACIÓN COMPARATIVA (NORMAL vs TTA) ---
def compare_tta_results(loader, idx_to_check=0):
    images, masks = next(iter(loader))
    
    batch_size = images.shape[0]
    idx = idx_to_check % batch_size  # evita IndexError
    
    img = images[idx:idx+1]
    mask_true = masks[idx]
    
    model.eval()
    with torch.no_grad():
        pred_std = torch.argmax(model(img.to(DEVICE)), dim=1).cpu()[0]
        
    pred_tta = predict_with_tta(model, img)[0]
    
    # Visualizar
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    
    img_np = img[0].permute(1, 2, 0).numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    
    ax[0].imshow(np.clip(img_np, 0, 1))
    ax[0].set_title("Imagen Original")
    
    ax[1].imshow(mask_true, cmap='nipy_spectral', vmin=0, vmax=5)
    ax[1].set_title("Realidad")
    
    ax[2].imshow(pred_std, cmap='nipy_spectral', vmin=0, vmax=5)
    ax[2].set_title("Predicción Normal")
    
    ax[3].imshow(pred_tta, cmap='nipy_spectral', vmin=0, vmax=5)
    ax[3].set_title("Predicción con TTA")
    
    plt.show()


# Ejecuta esto varias veces para ver diferentes ejemplos, EN LA 4 HAY PROBLEMAS
compare_tta_results(val_loader, idx_to_check=4)

In [ ]:
# ============================================
# 15. PRUEBA CON IMAGEN PROPIA (img_pruebas)
# ============================================
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

def test_single_image(image_path, model, device):
    # 1. Verificar si el archivo existe
    if not os.path.exists(image_path):
        print(f"❌ Error: No encuentro la imagen en {image_path}")
        return

    # 2. Cargar imagen original
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        print("❌ Error al leer la imagen. Verifica el formato.")
        return
        
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    
    # Guardamos una copia original para visualizar después (quizás redimensionada)
    original_h, original_w = image_rgb.shape[:2]

    # 3. Pre-procesamiento (IGUAL QUE EN EL ENTRENAMIENTO)
    # Debemos redimensionar a 256x256 porque así aprendió el modelo
    transform = A.Compose([
        A.Resize(384, 384),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    transformed = transform(image=image_rgb)
    img_tensor = transformed['image'].unsqueeze(0).to(device) # Añadir batch dim [1, 3, 256, 256]

    # 4. Predicción
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        # Argmax para obtener la clase con mayor probabilidad por pixel
        pred_mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()

    # 5. Visualización
    # Recuperamos la imagen redimensionada a 256x256 para mostrarla cuadrada
    # (Deshacemos la normalización para que se vean los colores bien)
    img_vis = img_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    img_vis = (img_vis * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img_vis = np.clip(img_vis, 0, 1)

    # Crear máscara de colores
    mask_rgb = np.zeros((384, 384, 3))
    
    # Usamos tus colores globales
    for cls_idx, color in enumerate(CLASS_COLORS):
        mask_rgb[pred_mask == cls_idx] = [c/255.0 for c in color]

    # 6. Plotear
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # A. Imagen (Redimensionada)
    axes[0].imshow(img_vis)
    axes[0].set_title(f"Imagen: {os.path.basename(image_path)}")
    axes[0].axis('off')
    
    # B. Predicción
    axes[1].imshow(mask_rgb)
    axes[1].set_title("Segmentación Predicha")
    axes[1].axis('off')
    
    # C. Superposición
    axes[2].imshow(img_vis)
    axes[2].imshow(mask_rgb, alpha=0.5)
    axes[2].set_title("Overlay")
    axes[2].axis('off')

    # Leyenda
    legend_patches = [mpatches.Patch(color=[c/255.0 for c in color], label=name) 
                      for name, color in zip(CLASS_NAMES, CLASS_COLORS)]
    axes[2].legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

# --- EJECUTAR PRUEBA ---
ruta_imagen = "img_pruebas/persona.jpg"

# Nos aseguramos de cargar el mejor modelo antes de probar
try:
    model.load_state_dict(torch.load("mejor_modelo_384_fixed_body.pth"))
    print("✅ Pesos del mejor modelo cargados.")
except:
    print("⚠️ Usando los pesos actuales en memoria (quizás no sean los mejores).")

test_single_image(ruta_imagen, model, DEVICE)

In [ ]:
# ============================================
# 16. PREDICCIÓN CON REESCALADO (SOLUCIÓN FINAL)
# ============================================
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

def predict_and_resize_back(image_path, model, device):
    if not os.path.exists(image_path):
        print(f"❌ Error: No encuentro {image_path}")
        return

    # 1. Cargar imagen original (Alta Resolución)
    image_bgr = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    
    # Guardamos dimensiones originales
    orig_h, orig_w = image_rgb.shape[:2]
    
    # 2. Pre-procesar para el modelo (Resize a 256x256)
    # Es OBLIGATORIO bajarla a 256 para que el modelo entienda el contexto
    transform = A.Compose([
        A.Resize(384, 384),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    transformed = transform(image=image_rgb)
    img_tensor = transformed['image'].unsqueeze(0).to(device)

    # 3. Predicción (El cerebro trabaja en 256x256)
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        # Obtenemos la máscara pequeña (256x256)
        pred_mask_small = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)

    # 4. POST-PROCESAMIENTO: Estirar la máscara al tamaño original
    # Usamos INTER_NEAREST para que los bordes de clase sean nítidos (sin difuminar)
    pred_mask_hd = cv2.resize(pred_mask_small, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    # 5. Visualización
    # Crear imagen de máscara coloreada en HD
    mask_rgb_hd = np.zeros((orig_h, orig_w, 3))
    
    for cls_idx, color in enumerate(CLASS_COLORS):
        mask_rgb_hd[pred_mask_hd == cls_idx] = [c/255.0 for c in color]

    # Plotear Resultados
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # A. Imagen Original Limpia
    axes[0].imshow(image_rgb)
    axes[0].set_title(f"Original ({orig_w}x{orig_h})")
    axes[0].axis('off')
    
    # B. Superposición HD
    axes[1].imshow(image_rgb)
    axes[1].imshow(mask_rgb_hd, alpha=0.6) # Transparencia para ver la ropa debajo
    axes[1].set_title("Segmentación Ajustada al Tamaño Real")
    axes[1].axis('off')

    # Leyenda
    legend_patches = [mpatches.Patch(color=[c/255.0 for c in color], label=name) 
                      for name, color in zip(CLASS_NAMES, CLASS_COLORS)]
    axes[1].legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left', title="Clases")

    plt.tight_layout()
    plt.show()

# --- EJECUTAR ---
predict_and_resize_back("img_pruebas/vestido_final.jpg", model, DEVICE)


In [ ]:
import os
import cv2
import numpy as np
import torch
import albumentations as A
from IPython.display import display as ipy_display
from PIL import Image as PILImage
import matplotlib.pyplot as plt

def extract_and_save_garments(image_path, model, class_names=CLASS_NAMES, out_dir="extracted_by_class", device=DEVICE):
    if not os.path.exists(image_path):
        print(f"❌ No existe {image_path}")
        return

    base = os.path.splitext(os.path.basename(image_path))[0]
    image_folder = os.path.join(out_dir, base)
    os.makedirs(image_folder, exist_ok=True)
    for cn in class_names:
        os.makedirs(os.path.join(image_folder, cn), exist_ok=True)

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("❌ Error leyendo la imagen.")
        return
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    transform = A.Compose([
        A.Resize(IMG_HEIGHT, IMG_WIDTH),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    tensor = transform(image=img_rgb)['image'].unsqueeze(0)

    model.eval()
    with torch.no_grad():
        logits = model(tensor.to(device))
        logits_f = model(torch.flip(tensor, dims=[3]).to(device))
        logits_f = torch.flip(logits_f, dims=[3])
        logits_avg = (logits + logits_f) / 2.0
        pred_small = torch.argmax(logits_avg, dim=1).squeeze().cpu().numpy().astype(np.uint8)

    pred_hd = cv2.resize(pred_small, (w, h), interpolation=cv2.INTER_NEAREST)

    saved = {}
    saved_masks = {}
    for cls_idx, cls_name in enumerate(class_names):
        if cls_idx == 0:
            saved[cls_name] = None
            saved_masks[cls_name] = None
            continue
        mask_bin = (pred_hd == cls_idx).astype(np.uint8)
        if mask_bin.sum() == 0:
            saved[cls_name] = None
            saved_masks[cls_name] = None
            continue

        # Prepare BGRA for saving (cv2 expects BGR order in first 3 channels)
        rgba = np.zeros((h, w, 4), dtype=np.uint8)
        rgba[..., :3] = img_rgb[..., ::-1]  # RGB -> BGR for cv2.imwrite
        rgba[..., 3] = (mask_bin * 255).astype(np.uint8)

        out_path = os.path.join(image_folder, cls_name, f"{base}_{cls_name}.png")
        cv2.imwrite(out_path, rgba)
        saved[cls_name] = out_path
        saved_masks[cls_name] = mask_bin

    # Colored overlay for whole image
    mask_rgb_hd = np.zeros((h, w, 3), dtype=np.float32)
    for idx, color in enumerate(CLASS_COLORS):
        if idx == 0:
            continue
        mask_rgb_hd[pred_hd == idx] = np.array(color) / 255.0

    overlay = (img_rgb.astype(np.float32) / 255.0 * 0.6 + mask_rgb_hd * 0.4)
    overlay = (np.clip(overlay, 0, 1) * 255).astype(np.uint8)
    overlay_path = os.path.join(image_folder, f"{base}_overlay.png")
    cv2.imwrite(overlay_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    print(f"✅ Resultados para: {base}")
    for k, v in saved.items():
        print(f" - {k}: {v}")
    print(f"Overlay guardado en: {overlay_path}")

    # Mostrar en notebook: Original + columnas con cada clase (body, footwear, lowerwear, topwear)
    display_order = ['body', 'footwear', 'lowerwear', 'topwear']
    cols = 1 + len(display_order)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 6))
    if cols == 1:
        axes = [axes]

    axes[0].imshow(img_rgb)
    axes[0].set_title("Original")
    axes[0].axis('off')

    for i, cls in enumerate(display_order, start=1):
        ax = axes[i]
        mask = saved_masks.get(cls)
        if mask is None:
            blank = np.ones((h, w, 3), dtype=np.uint8) * 255
            ax.imshow(blank)
            ax.set_title(f"{cls}\n(no detectado)")
        else:
            # Create an RGBA image for visualization so the segmented part appears in its column
            rgb = img_rgb.copy()  # RGB uint8
            alpha_u8 = (mask * 255).astype(np.uint8)
            rgba_vis = np.dstack([rgb, alpha_u8])  # uint8 RGBA (R,G,B,A)
            ax.imshow(rgba_vis)
            ax.set_title(cls)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    print("\nContenido guardado en:", image_folder)
    for cls in class_names:
        cls_folder = os.path.join(image_folder, cls)
        if os.path.exists(cls_folder):
            files = sorted([f for f in os.listdir(cls_folder) if f.lower().endswith('.png')])
            if files:
                print(f"\n{cls}:")
                for f in files:
                    p = os.path.join(cls_folder, f)
                    print(" ", p)
        else:
            print(f"\n{cls}: (no existe)")

    return {"by_class": saved, "overlay": overlay_path}

# Ejecutar con img_pruebas/persona.jpg si existe, si no usar ruta_imagen
test_image = "img_pruebas/pondoy.jpg"
image_to_use = test_image if os.path.exists(test_image) else ruta_imagen
result_paths = extract_and_save_garments(image_to_use, model, CLASS_NAMES, out_dir="extracted_by_class", device=DEVICE)
print(result_paths)


# Implementacion de YOLO

pip install ultralytics

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# Cargar modelo YOLO
yolo = YOLO("yolov8n.pt")

img = cv2.imread("img_pruebas/pondoy.jpg")
img_draw = img.copy()

results = yolo(img)

detectado = False

for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        if cls == 0:  # persona
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Dibujar rectángulo
            cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 255, 0), 3)

            # Texto
            cv2.putText(img_draw, "Persona", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

            detectado = True
            break
    if detectado:
        break

# Mostrar imagen completa
plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(img_draw, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

if not detectado:
    print("No se detectó ninguna persona.")


In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# ------------------------
# CONFIG
# ------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 5

CLASS_COLORS = [
    (0,0,0),        # 0 background
    (220,20,60),    # 1 topwear
    (30,144,255),   # 2 lowerwear
    (138,43,226),   # 3 footwear
    (46,139,87),    # 4 body
]

# ------------------------
# LOAD U-NET++
# ------------------------
model = smp.UnetPlusPlus(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES,
    decoder_attention_type="scse"
).to(DEVICE)

model.load_state_dict(torch.load("mejor_modelo_384_fixed_body.pth", map_location=DEVICE))
model.eval()

# ------------------------
# TRANSFORM
# ------------------------
transform = A.Compose([
    A.Resize(384, 384),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ------------------------
# SEGMENT FUNCTION
# ------------------------
def segment_crop(crop_bgr):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    aug = transform(image=crop_rgb)["image"]
    x = aug.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

    return pred  # 384x384

# ------------------------
# MAIN
# ------------------------
yolo = YOLO("yolov8n.pt")

img = cv2.imread("img_pruebas/personas.jpg")
img_out = img.copy()

results = yolo(img)

person_id = 0

for r in results:
    for box in r.boxes:
        if int(box.cls[0]) != 0:
            continue

        person_id += 1

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # ---- PAD ----
        h, w, _ = img.shape
        pad = 0.25

        dx = int((x2 - x1) * pad)
        dy = int((y2 - y1) * pad)

        x1p = max(0, x1 - dx)
        y1p = max(0, y1 - dy)
        x2p = min(w, x2 + dx)
        y2p = min(h, y2 + dy)

        crop = img[y1p:y2p, x1p:x2p]

        # ---- SEGMENT ----
        mask_384 = segment_crop(crop)

        # ---- RESIZE BACK ----
        mask_crop = cv2.resize(
            mask_384.astype(np.uint8),
            (x2p - x1p, y2p - y1p),
            interpolation=cv2.INTER_NEAREST
        )

        # ---- COLORIZE ----
        overlay = crop.copy()
        for cls_idx, color in enumerate(CLASS_COLORS):
            overlay[mask_crop == cls_idx] = color

        blended = cv2.addWeighted(crop, 0.5, overlay, 0.5, 0)

        # ---- PUT BACK ----
        img_out[y1p:y2p, x1p:x2p] = blended

        # ---- DRAW BOX + ID ----
        cv2.rectangle(img_out, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img_out, f"ID {person_id}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

# ------------------------
# SHOW
# ------------------------
plt.figure(figsize=(10,10))
plt.imshow(cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
